# 00 — Shared Pipeline

Runs once per video, produces a merged per-frame DataFrame shared by all downstream metric notebooks.

**Extractors:**
- **Py-Feat** `Detector` → AUs, emotions, head pose (pitch/yaw/roll)
- **MediaPipe Face Mesh** (with iris refinement) → iris + eye-corner landmarks
- **MediaPipe Pose** → body landmarks (shoulders, wrists, etc.)
- **MediaPipe Hands** → hand landmarks

**Output:** `data/<video_stem>_merged.parquet` indexed by frame, with a `timestamp` column (seconds).

All three MediaPipe models share a single `cv2.VideoCapture` loop for efficiency.

In [12]:
import os
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import mediapipe as mp
from feat import Detector

pd.set_option("display.max_columns", 50)

## Config

Set `VIDEO_PATH` to the file you want to process. `SKIP_FRAMES=N` processes every Nth frame (1 = every frame). Py-Feat and MediaPipe both honor this so their frame indices stay aligned.

In [13]:
VIDEO_PATH = "../player2.mp4"  # <-- change me
SKIP_FRAMES = 5

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)

video_stem = Path(VIDEO_PATH).stem
OUTPUT_PATH = DATA_DIR / f"{video_stem}_merged.parquet"
print(f"Video:  {VIDEO_PATH}")
print(f"Output: {OUTPUT_PATH}")

Video:  ../player2.mp4
Output: /Users/atharvumap/Documents/Projects/PyfeatTesting/data/player2_merged.parquet


In [14]:
# Grab video metadata once
cap = cv2.VideoCapture(VIDEO_PATH)
assert cap.isOpened(), f"Could not open {VIDEO_PATH}"
FPS = cap.get(cv2.CAP_PROP_FPS)
N_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
WIDTH = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
HEIGHT = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()
print(f"{WIDTH}x{HEIGHT} @ {FPS:.2f} fps, {N_FRAMES} frames ({N_FRAMES/FPS:.1f}s)")

1278x726 @ 60.00 fps, 1983 frames (33.0s)


## 1. Py-Feat

In [15]:
detector = Detector()
fex = detector.detect_video(VIDEO_PATH, skip_frames=SKIP_FRAMES)
print(fex.shape)
fex.head(3)

100%|██████████| 397/397 [07:48<00:00,  1.18s/it]


(397, 687)


,FaceRectX,FaceRectY,FaceRectWidth,FaceRectHeight,FaceScore,x_0,x_1,x_2,x_3,x_4,x_5,x_6,x_7,x_8,x_9,x_10,x_11,x_12,x_13,x_14,x_15,x_16,x_17,x_18,x_19,...,Identity_491,Identity_492,Identity_493,Identity_494,Identity_495,Identity_496,Identity_497,Identity_498,Identity_499,Identity_500,Identity_501,Identity_502,Identity_503,Identity_504,Identity_505,Identity_506,Identity_507,Identity_508,Identity_509,Identity_510,Identity_511,Identity_512,input,frame,approx_time
frame,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,343.494350,467.830665,226.699256,278.963581,0.998191,342.706928,342.805915,345.572118,353.101065,366.867264,387.728386,412.876297,439.443927,468.385348,497.612986,523.891725,546.625070,565.723173,577.280207,581.167796,581.009070,578.522742,367.703036,385.456255,407.014835,...,-0.054157,-0.032872,-0.035858,0.080359,0.032219,-0.050250,0.005115,0.016696,0.049794,0.071881,0.038805,-0.053388,-0.036022,-0.028923,-0.036742,-0.002044,-0.028929,0.045117,0.059097,-0.028450,0.012425,0.053851,../player2.mp4,0,00:00
5,343.597085,469.456884,226.209896,277.291228,0.998224,340.584949,341.263178,344.516696,352.156914,365.637745,385.976465,410.825620,437.169313,466.114505,495.399374,521.844435,544.557106,563.435184,574.680670,578.496480,578.554533,576.233012,367.435128,385.583237,406.951778,...,-0.049839,-0.034398,-0.039223,0.081450,0.032954,-0.048959,0.010737,0.018597,0.049972,0.064762,0.039537,-0.047433,-0.031050,-0.031687,-0.031104,-0.003421,-0.034163,0.049822,0.058075,-0.029426,0.012544,0.057795,../player2.mp4,5,00:00
10,343.547301,468.602790,226.252066,278.022590,0.998231,341.629348,341.687176,344.473742,352.084308,365.883430,386.639941,411.642552,438.128379,467.065769,496.305122,522.593117,545.345608,564.412797,575.938702,579.772849,579.555975,577.041505,367.106944,384.767889,406.249742,...,-0.054022,-0.034052,-0.035616,0.079463,0.032190,-0.051289,0.005163,0.016727,0.048979,0.071963,0.038964,-0.053797,-0.035491,-0.027545,-0.035773,-0.002886,-0.028918,0.045870,0.058983,-0.028847,0.012594,0.052741,../player2.mp4,10,00:00


In [16]:
# Normalize Py-Feat output to a plain DataFrame with an integer 'frame' column,
# then prefix every non-frame column with 'pf_' so it won't collide with MediaPipe columns on merge.
pf_raw = pd.DataFrame(fex)

# Locate the frame info. Py-Feat can put it in the index (named or unnamed) or as a column.
if "frame" in pf_raw.columns:
    frame_series = pf_raw["frame"]
    pf = pf_raw.drop(columns=["frame"]).reset_index(drop=True)
elif pf_raw.index.name == "frame":
    frame_series = pf_raw.index.to_series().reset_index(drop=True)
    pf = pf_raw.reset_index(drop=True)
else:
    # Fall back: assume the (possibly unnamed) index is the frame counter.
    frame_series = pf_raw.index.to_series().reset_index(drop=True)
    pf = pf_raw.reset_index(drop=True)

pf = pf.add_prefix("pf_")
pf.insert(0, "frame", pd.to_numeric(frame_series.values, errors="coerce"))
pf = pf.dropna(subset=["frame"]).copy()
pf["frame"] = pf["frame"].astype(int)
print(pf.shape)
pf.head(3)


(397, 687)


,frame,pf_FaceRectX,pf_FaceRectY,pf_FaceRectWidth,pf_FaceRectHeight,pf_FaceScore,pf_x_0,pf_x_1,pf_x_2,pf_x_3,pf_x_4,pf_x_5,pf_x_6,pf_x_7,pf_x_8,pf_x_9,pf_x_10,pf_x_11,pf_x_12,pf_x_13,pf_x_14,pf_x_15,pf_x_16,pf_x_17,pf_x_18,...,pf_Identity_490,pf_Identity_491,pf_Identity_492,pf_Identity_493,pf_Identity_494,pf_Identity_495,pf_Identity_496,pf_Identity_497,pf_Identity_498,pf_Identity_499,pf_Identity_500,pf_Identity_501,pf_Identity_502,pf_Identity_503,pf_Identity_504,pf_Identity_505,pf_Identity_506,pf_Identity_507,pf_Identity_508,pf_Identity_509,pf_Identity_510,pf_Identity_511,pf_Identity_512,pf_input,pf_approx_time
0,0,343.494350,467.830665,226.699256,278.963581,0.998191,342.706928,342.805915,345.572118,353.101065,366.867264,387.728386,412.876297,439.443927,468.385348,497.612986,523.891725,546.625070,565.723173,577.280207,581.167796,581.009070,578.522742,367.703036,385.456255,...,-0.043177,-0.054157,-0.032872,-0.035858,0.080359,0.032219,-0.050250,0.005115,0.016696,0.049794,0.071881,0.038805,-0.053388,-0.036022,-0.028923,-0.036742,-0.002044,-0.028929,0.045117,0.059097,-0.028450,0.012425,0.053851,../player2.mp4,00:00
1,5,343.597085,469.456884,226.209896,277.291228,0.998224,340.584949,341.263178,344.516696,352.156914,365.637745,385.976465,410.825620,437.169313,466.114505,495.399374,521.844435,544.557106,563.435184,574.680670,578.496480,578.554533,576.233012,367.435128,385.583237,...,-0.038491,-0.049839,-0.034398,-0.039223,0.081450,0.032954,-0.048959,0.010737,0.018597,0.049972,0.064762,0.039537,-0.047433,-0.031050,-0.031687,-0.031104,-0.003421,-0.034163,0.049822,0.058075,-0.029426,0.012544,0.057795,../player2.mp4,00:00
2,10,343.547301,468.602790,226.252066,278.022590,0.998231,341.629348,341.687176,344.473742,352.084308,365.883430,386.639941,411.642552,438.128379,467.065769,496.305122,522.593117,545.345608,564.412797,575.938702,579.772849,579.555975,577.041505,367.106944,384.767889,...,-0.042190,-0.054022,-0.034052,-0.035616,0.079463,0.032190,-0.051289,0.005163,0.016727,0.048979,0.071963,0.038964,-0.053797,-0.035491,-0.027545,-0.035773,-0.002886,-0.028918,0.045870,0.058983,-0.028847,0.012594,0.052741,../player2.mp4,00:00


## 2. MediaPipe (Face Mesh + Pose + Hands)

Single video-capture pass; all three solutions run on each sampled frame.

In [17]:
# Landmark indices we care about.
# Face Mesh iris (requires refine_landmarks=True): 468-472 left, 473-477 right.
# Left iris center: 468, right iris center: 473.
# Eye corners (outer, inner): left 33/133, right 263/362.
FACE_POINTS = {
    "l_iris": 468, "r_iris": 473,
    "l_eye_outer": 33, "l_eye_inner": 133,
    "r_eye_inner": 362, "r_eye_outer": 263,
    "l_eye_top": 159, "l_eye_bot": 145,
    "r_eye_top": 386, "r_eye_bot": 374,
}
# Pose landmarks we care about (MediaPipe Pose indexing).
POSE_POINTS = {
    "nose": 0,
    "l_shoulder": 11, "r_shoulder": 12,
    "l_elbow": 13, "r_elbow": 14,
    "l_wrist": 15, "r_wrist": 16,
    "l_hip": 23, "r_hip": 24,
}

In [18]:
def run_mediapipe(video_path: str, skip_frames: int = 1) -> pd.DataFrame:
    """Return a per-frame DataFrame of MediaPipe landmarks (normalized 0-1 coords)."""
    mp_face = mp.solutions.face_mesh.FaceMesh(
        static_image_mode=False, max_num_faces=1,
        refine_landmarks=True, min_detection_confidence=0.5,
    )
    mp_pose = mp.solutions.pose.Pose(
        static_image_mode=False, model_complexity=1, min_detection_confidence=0.5,
    )
    mp_hands = mp.solutions.hands.Hands(
        static_image_mode=False, max_num_hands=2, min_detection_confidence=0.5,
    )

    cap = cv2.VideoCapture(video_path)
    rows = []
    frame_idx = 0
    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            if frame_idx % skip_frames != 0:
                frame_idx += 1
                continue
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            row = {"frame": frame_idx}

            face_res = mp_face.process(rgb)
            if face_res.multi_face_landmarks:
                lms = face_res.multi_face_landmarks[0].landmark
                for name, idx in FACE_POINTS.items():
                    row[f"mp_face_{name}_x"] = lms[idx].x
                    row[f"mp_face_{name}_y"] = lms[idx].y
                    row[f"mp_face_{name}_z"] = lms[idx].z

            pose_res = mp_pose.process(rgb)
            if pose_res.pose_landmarks:
                lms = pose_res.pose_landmarks.landmark
                for name, idx in POSE_POINTS.items():
                    row[f"mp_pose_{name}_x"] = lms[idx].x
                    row[f"mp_pose_{name}_y"] = lms[idx].y
                    row[f"mp_pose_{name}_z"] = lms[idx].z
                    row[f"mp_pose_{name}_v"] = lms[idx].visibility

            hands_res = mp_hands.process(rgb)
            if hands_res.multi_hand_landmarks and hands_res.multi_handedness:
                for hlm, handed in zip(hands_res.multi_hand_landmarks, hands_res.multi_handedness):
                    label = handed.classification[0].label.lower()  # 'left' or 'right'
                    for i, lm in enumerate(hlm.landmark):
                        row[f"mp_hand_{label}_{i}_x"] = lm.x
                        row[f"mp_hand_{label}_{i}_y"] = lm.y
                        row[f"mp_hand_{label}_{i}_z"] = lm.z

            rows.append(row)
            frame_idx += 1
    finally:
        cap.release()
        mp_face.close()
        mp_pose.close()
        mp_hands.close()
    return pd.DataFrame(rows)

In [19]:
mp_df = run_mediapipe(VIDEO_PATH, skip_frames=SKIP_FRAMES)
print(mp_df.shape)
mp_df.head(3)

I0000 00:00:1777005649.420685 14734139 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 89.3), renderer: Apple M4 Pro
W0000 00:00:1777005649.422567 14899310 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1777005649.425465 14734139 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 89.3), renderer: Apple M4 Pro
W0000 00:00:1777005649.426391 14899307 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1777005649.428916 14734139 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 89.3), renderer: Apple M4 Pro
W0000 00:00:1777005649.433508 14899332 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1777005649.439031 14899334 inference_feedback_manager.cc:114] Feedback manager requires 

(397, 130)


,frame,mp_face_l_iris_x,mp_face_l_iris_y,mp_face_l_iris_z,mp_face_r_iris_x,mp_face_r_iris_y,mp_face_r_iris_z,mp_face_l_eye_outer_x,mp_face_l_eye_outer_y,mp_face_l_eye_outer_z,mp_face_l_eye_inner_x,mp_face_l_eye_inner_y,mp_face_l_eye_inner_z,mp_face_r_eye_inner_x,mp_face_r_eye_inner_y,mp_face_r_eye_inner_z,mp_face_r_eye_outer_x,mp_face_r_eye_outer_y,mp_face_r_eye_outer_z,mp_face_l_eye_top_x,mp_face_l_eye_top_y,mp_face_l_eye_top_z,mp_face_l_eye_bot_x,mp_face_l_eye_bot_y,mp_face_l_eye_bot_z,...,mp_hand_right_12_z,mp_hand_right_13_x,mp_hand_right_13_y,mp_hand_right_13_z,mp_hand_right_14_x,mp_hand_right_14_y,mp_hand_right_14_z,mp_hand_right_15_x,mp_hand_right_15_y,mp_hand_right_15_z,mp_hand_right_16_x,mp_hand_right_16_y,mp_hand_right_16_z,mp_hand_right_17_x,mp_hand_right_17_y,mp_hand_right_17_z,mp_hand_right_18_x,mp_hand_right_18_y,mp_hand_right_18_z,mp_hand_right_19_x,mp_hand_right_19_y,mp_hand_right_19_z,mp_hand_right_20_x,mp_hand_right_20_y,mp_hand_right_20_z
0,0,0.322408,0.810166,-0.004750,0.397885,0.800686,0.000592,0.302788,0.810404,0.000732,0.340781,0.812322,-0.003313,0.382909,0.805999,0.000097,0.415665,0.797318,0.007898,0.318885,0.806768,-0.008977,0.321619,0.817058,-0.005894,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,5,0.322310,0.810220,-0.002419,0.398273,0.800394,0.002635,0.303515,0.810470,0.002936,0.341009,0.812546,-0.001055,0.383469,0.806058,0.002095,0.414680,0.796566,0.009779,0.319257,0.806197,-0.006298,0.322288,0.817005,-0.003770,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,10,0.322283,0.810717,-0.002499,0.398085,0.800240,0.001935,0.303901,0.810894,0.002971,0.340662,0.813295,-0.001264,0.383355,0.806027,0.001492,0.414480,0.796555,0.008979,0.319301,0.806108,-0.006399,0.322151,0.818119,-0.003815,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Merge and save

In [20]:
merged = pf.merge(mp_df, on="frame", how="outer").sort_values("frame").reset_index(drop=True)
merged["timestamp"] = merged["frame"] / FPS

# Attach video-level metadata as parquet attrs via a sidecar dict (store as string col if needed downstream).
print(f"Merged shape: {merged.shape}")
print(f"Frames: {merged['frame'].min()} → {merged['frame'].max()}")
print(f"Duration covered: {merged['timestamp'].max():.2f}s")
merged.head(3)

Merged shape: (397, 817)
Frames: 0 → 1980
Duration covered: 33.00s


,frame,pf_FaceRectX,pf_FaceRectY,pf_FaceRectWidth,pf_FaceRectHeight,pf_FaceScore,pf_x_0,pf_x_1,pf_x_2,pf_x_3,pf_x_4,pf_x_5,pf_x_6,pf_x_7,pf_x_8,pf_x_9,pf_x_10,pf_x_11,pf_x_12,pf_x_13,pf_x_14,pf_x_15,pf_x_16,pf_x_17,pf_x_18,...,mp_hand_right_13_x,mp_hand_right_13_y,mp_hand_right_13_z,mp_hand_right_14_x,mp_hand_right_14_y,mp_hand_right_14_z,mp_hand_right_15_x,mp_hand_right_15_y,mp_hand_right_15_z,mp_hand_right_16_x,mp_hand_right_16_y,mp_hand_right_16_z,mp_hand_right_17_x,mp_hand_right_17_y,mp_hand_right_17_z,mp_hand_right_18_x,mp_hand_right_18_y,mp_hand_right_18_z,mp_hand_right_19_x,mp_hand_right_19_y,mp_hand_right_19_z,mp_hand_right_20_x,mp_hand_right_20_y,mp_hand_right_20_z,timestamp
0,0,343.494350,467.830665,226.699256,278.963581,0.998191,342.706928,342.805915,345.572118,353.101065,366.867264,387.728386,412.876297,439.443927,468.385348,497.612986,523.891725,546.625070,565.723173,577.280207,581.167796,581.009070,578.522742,367.703036,385.456255,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000
1,5,343.597085,469.456884,226.209896,277.291228,0.998224,340.584949,341.263178,344.516696,352.156914,365.637745,385.976465,410.825620,437.169313,466.114505,495.399374,521.844435,544.557106,563.435184,574.680670,578.496480,578.554533,576.233012,367.435128,385.583237,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.083333
2,10,343.547301,468.602790,226.252066,278.022590,0.998231,341.629348,341.687176,344.473742,352.084308,365.883430,386.639941,411.642552,438.128379,467.065769,496.305122,522.593117,545.345608,564.412797,575.938702,579.772849,579.555975,577.041505,367.106944,384.767889,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.166667


In [21]:
merged.to_parquet(OUTPUT_PATH, index=False)

# Write a small sidecar with video metadata so downstream notebooks know FPS etc.
meta = {
    "video_path": str(Path(VIDEO_PATH).resolve()),
    "fps": float(FPS),
    "n_frames": int(N_FRAMES),
    "width": int(WIDTH),
    "height": int(HEIGHT),
    "skip_frames": int(SKIP_FRAMES),
    "effective_fps": float(FPS / SKIP_FRAMES),
}
pd.Series(meta).to_json(OUTPUT_PATH.with_suffix(".meta.json"))
print(f"Saved: {OUTPUT_PATH}")
print(f"Saved: {OUTPUT_PATH.with_suffix('.meta.json')}")

Saved: /Users/atharvumap/Documents/Projects/PyfeatTesting/data/player2_merged.parquet
Saved: /Users/atharvumap/Documents/Projects/PyfeatTesting/data/player2_merged.meta.json


## Sanity check

Quick look at what's in the merged frame. Downstream notebooks should start by loading this parquet + its sidecar json.

In [22]:
loaded = pd.read_parquet(OUTPUT_PATH)
meta_loaded = pd.read_json(OUTPUT_PATH.with_suffix(".meta.json"), typ="series")
print("meta:", dict(meta_loaded))
print("columns (sample):", [c for c in loaded.columns[:20]])
print("n rows:", len(loaded))

meta: {'video_path': '/Users/atharvumap/Documents/Projects/PyfeatTesting/player2.mp4', 'fps': 60.0, 'n_frames': 1983, 'width': 1278, 'height': 726, 'skip_frames': 5, 'effective_fps': 12.0}
columns (sample): ['frame', 'pf_FaceRectX', 'pf_FaceRectY', 'pf_FaceRectWidth', 'pf_FaceRectHeight', 'pf_FaceScore', 'pf_x_0', 'pf_x_1', 'pf_x_2', 'pf_x_3', 'pf_x_4', 'pf_x_5', 'pf_x_6', 'pf_x_7', 'pf_x_8', 'pf_x_9', 'pf_x_10', 'pf_x_11', 'pf_x_12', 'pf_x_13']
n rows: 397
